# Study 962 — Do It Yourself 🧰

**A sector fund is mostly a handful of mega-caps. So why not just buy them and skip the fee?**

The Technology Select Sector SPDR keeps about two thirds of its money in ten names. With
zero-commission trading and fractional shares, the forum argument writes itself: hold the
ten, skip the **0.08%/yr** expense ratio, keep the difference.

We build that basket for **XLK**, **XLE** and **XLF** at depths 3 / 5 / 10, rebalance it
monthly with a one-day execution lag, charge 5 bps one-way on every trade, and race it
against the fund it is meant to replace. Daily total-return closes, 48 tickers,
2011-01-03 → 2026-06-30 (3,895 days).

*Real-tape numbers below are the frozen headline (`docs/results.md`, Fingerprint
`1f3a3deb14a7`); the live cells run only the offline synthetic control. As-of 2026-06-30.*


## 1. The argument, and the trap inside it

The argument is arithmetically fine *if* the ten names behave like the fund. The trap is in the phrase "the ten names". Which ten? If you look up the holdings today and run them backwards, you have quietly given your 2011 self a list of the companies that would go on to dominate the next fifteen years.

So we run it both ways: **hindsight** (today's published weights, applied from 2011) and **contemporaneous** (the top ten as published in January 2011, equal weight, fixed from day one). Same machinery, same costs, same lag.

In [1]:
R = dict(look_gap=6.08, look_t=6.1, blend_gap=-0.76, blend_t=-0.58, eq2026_gap=5.06,
         raw_premium=6.84, vintage_premium=5.81, weighting_premium=1.02,
         blend_te=5.13, fee=0.08)
print('hindsight basket      : %+.2f%% a year vs the fund   (t = %+.2f)'
      % (R['look_gap'], R['look_t']))
print('contemporaneous basket: %+.2f%% a year vs the fund   (t = %+.2f)'
      % (R['blend_gap'], R['blend_t']))
print()
print('raw difference between them          : %+.2f%% a year' % R['raw_premium'])
print('  of which knowing the names (look-ahead): %+.2f%%' % R['vintage_premium'])
print('  of which just cap- vs equal-weighting  : %+.2f%%' % R['weighting_premium'])

hindsight basket      : +6.08% a year vs the fund   (t = +6.10)
contemporaneous basket: -0.76% a year vs the fund   (t = -0.58)

raw difference between them          : +6.84% a year
  of which knowing the names (look-ahead): +5.81%
  of which just cap- vs equal-weighting  : +1.02%


> ⚖️ **Why the split matters.** The hindsight basket changes **two** things at once — *which* names, and *how* they are weighted. So we ran a third basket: today's names at **equal** weight. That pins the weighting scheme and leaves only the hindsight, which is worth **+5.81%/yr** — not the raw +6.84%. The leftover is cap-weighting, and it is not a peek at the answer sheet.

## 2. What you were actually hunting

The prize is the expense ratio: **0.08% a year**. Eight basis points. The bill is the **tracking error** — how far your ten names wander from the fund in a typical year. Blended across the three sectors that is **5.13% a year**; inside technology and energy it is **9.1%** and **9.8%**.

That is not a rounding error on the fee. It is **64 times** the fee, and it points in both directions.

In [2]:
for tag, te in [('blended top-10', 5.13), ('XLK top-10', 9.13), ('XLE top-10', 9.75)]:
    print('%-15s tracking error %5.2f%%/yr  = %3.0fx the %.2f%% fee you saved'
          % (tag, te, te / 0.08, 0.08))

blended top-10  tracking error  5.13%/yr  =  64x the 0.08% fee you saved
XLK top-10      tracking error  9.13%/yr  = 114x the 0.08% fee you saved
XLE top-10      tracking error  9.75%/yr  = 122x the 0.08% fee you saved


> 🔬 **For the quants.** The standard error on the annualised gap is **1.31%/yr** — about **16×** the fee under test. At this tracking error you would need roughly **16,472 years** of tape for the fee saving to reach |*t*| = 2. The honest statement is not "the saving isn't there"; it is that no test of this shape can ever see it.

## 3. What 5% tracking error feels like, year by year

Averages hide the experience. Here is the gap between each do-it-yourself basket and its fund, calendar year by calendar year, in percentage points of total return. Double-digit misses in **both** directions are routine — you are not saving eight basis points, you are running an undiagnosed active bet.

In [3]:
years = [2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]
xlk = [6.9, -0.9, -2.8, -3.7, -4.2, 3.3, -12.8, 1.7, -10.2, -24.0, -7.6, 6.3, -20.5, -4.0, 8.6, -10.0]
xle = [-4.2, -4.3, 0.9, 2.2, -2.9, 5.1, -2.7, -6.6, -8.2, 3.8, 12.2, 10.8, -6.2, -20.2, -2.7, 0.4]
xlf = [-5.1, 5.3, 5.4, -1.6, -2.8, 0.2, -1.1, -2.0, 4.3, -6.8, 0.5, 0.5, 2.9, 7.7, 14.0, 8.2]
print('year    XLK     XLE     XLF')
for y, a, b, c in zip(years, xlk, xle, xlf):
    star = '  <-- double-digit miss' if min(a, b, c) <= -10 else ''
    print('%d  %+6.1f  %+6.1f  %+6.1f%s' % (y, a, b, c, star))
print('\n(2026 is a half-year, to 2026-06-30)')

year    XLK     XLE     XLF
2011    +6.9    -4.2    -5.1
2012    -0.9    -4.3    +5.3
2013    -2.8    +0.9    +5.4
2014    -3.7    +2.2    -1.6
2015    -4.2    -2.9    -2.8
2016    +3.3    +5.1    +0.2
2017   -12.8    -2.7    -1.1  <-- double-digit miss
2018    +1.7    -6.6    -2.0
2019   -10.2    -8.2    +4.3  <-- double-digit miss
2020   -24.0    +3.8    -6.8  <-- double-digit miss
2021    -7.6   +12.2    +0.5
2022    +6.3   +10.8    +0.5
2023   -20.5    -6.2    +2.9  <-- double-digit miss
2024    -4.0   -20.2    +7.7  <-- double-digit miss
2025    +8.6    -2.7   +14.0
2026   -10.0    +0.4    +8.2  <-- double-digit miss

(2026 is a half-year, to 2026-06-30)


## 4. And you inherit the concentration

Ten names are not a sector. The energy basket's worst twelve months cost **-25.7%** relative to simply owning XLE, and its worst drawdown was **-81.8%** against the fund's **-71.3%** — over ten percentage points deeper. None of that risk is paid for: the excess-of-cash Sharpe advantage over the fund is **-0.04** in tech, **-0.07** in energy, **+0.00** in financials.

## 5. Live check — the machinery is unbiased (offline synthetic)

Could the harness simply be blind to fees? No. On a synthetic sector world where the fund skims a deliberately fat 2%/yr, the same code recovers most of that fee with an enormous *t*. Switch the fee off and it recovers nothing. So the real-tape non-result is a fact about markets, not a bug.

In [4]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from diy_sector import data, strategy as st
p1, t1 = data.synthetic_panel(signal_strength=1.0, seed=962)
p0, t0 = data.synthetic_panel(signal_strength=0.0, seed=962)
d1 = st.synthetic_detect(p1, t1, cost_bps=0.0)
d0 = st.synthetic_detect(p0, t0, cost_bps=0.0)
print('fund skims %.2f%%/yr -> basket recovers %+.2f%%/yr (t = %+.2f)'
      % (t1['planted_fee_ann']*100, d1['ann_gap']*100, d1['t_diff']))
print('fund skims  0.00%%/yr -> basket recovers %+.2f%%/yr (t = %+.2f)'
      % (d0['ann_gap']*100, d0['t_diff']))

fund skims 2.00%/yr -> basket recovers +1.79%/yr (t = +11.27)
fund skims  0.00%/yr -> basket recovers -0.23%/yr (t = -1.47)


## Verdict

- **Signal — None.** With a contemporaneous holdings list the gap is **-0.76%/yr** (HAC *t* = -0.58), the confidence interval [-3.21%, +1.69%] straddles zero, and no sector clears |*t*| = 2. The only impressive number in the study — **+6.08%/yr, *t* = +6.10** — belongs to the hindsight basket, of which **+5.81%/yr** is pure look-ahead and +1.02%/yr is just cap-weighting.
- **Tradability — Mirage.** You would be swapping a certain **0.08%/yr** saving for **5.13%/yr** of two-sided tracking error, a worst twelve months of **-12.3%**, and a deeper drawdown in the sector where it hurt most. Buy the fund.